In [ ]:
%reset -f
import math
import os
import warnings
from collections.abc import Callable
from math import pow, sqrt

import forallpeople as units
import handcalcs
import handcalcs.render
import numpy as np

# import polars as pl
import pandas as pd
import sympy as sp
from structural_tools.notebook import (
    check_value,  # noqa: F401
    display_table,
    display_text,
    set_params_columns,
    sig_figs,  # noqa: F401
)
from structural_tools.seismic.asce.parameters import (
    RiskCategory,
    SeismicParameters,
    SiteClass,
)

units.environment(env_name="structural", top_level=True)
handcalcs.set_option("custom_symbols", {"star": "^*", "__": ","})
handcalcs.set_option("greek_exclusions", ["psi"])
handcalcs.set_option("display_precision", 4)
handcalcs.set_option("param_columns", 1)
handcalcs.set_option("latex_block_start", "\\begin{equation*}")
handcalcs.set_option("latex_block_end", "\\end{equation*}")

# Effective Seismic Weight Calculation

We calculate the effective seismic weight with unit weights and square footages.

## Unit Weights

Weights will be assigned with area unit weights.

In [2]:
set_params_columns(4)

In [3]:
%%render params
U_corridor = 35 / 1000 * ksf
U_floor = 35 / 1000 * ksf
U_roof = 25 / 1000 * ksf
U_wall = 12 / 1000 * ksf

<IPython.core.display.Latex object>

## Building Geometry

The lengths and widths of the weight areas are consistent across levels 2, 3, and 4.

In [4]:
set_params_columns(5)

In [5]:
%%render params
L_floor = 230 * ft
w_floor = (30 + 9 / 12) * ft
L_corridor = 230 * ft
w_corridor = 5.5 * ft
w_building = (w_floor * 2) + w_corridor

<IPython.core.display.Latex object>

The wall heights vary across the levels.

In [6]:
%%render params
h_wall__1 = 11.5 * ft
h_wall__2 = 10.5 * ft
h_wall__3 = 10.5 * ft
h_wall__4 = 10 * ft

<IPython.core.display.Latex object>

There is a small jog in the building floorplan. This does not affect the area, but the perimeter will be longer.

In [7]:
%%render params
L_jog = 5 * ft

<IPython.core.display.Latex object>

In [8]:
%%render
P_wall = (2 * L_floor) + (2 * w_building) + (2 * L_jog)

<IPython.core.display.Latex object>

Areas can now be calculated. Note that the areas of the walls are the actual wall areas; they have not been distributed to each floor based on tributary area, yet.

In [9]:
%%render 
A_floor = 2 * (L_floor * w_floor)
# This is a slight overestimate as the corridor doesn't stretch the full length of the building
A_corridor = L_corridor * w_corridor
A_level = L_floor * w_building
A_wall__1 = h_wall__1 * P_wall
A_wall__2 = h_wall__2 * P_wall
A_wall__3 = h_wall__3 * P_wall
A_wall__4 = h_wall__4 * P_wall

<IPython.core.display.Latex object>

## Component Weights

The typical floor and corridor weights are

In [10]:
%%render
W_floor__typ = U_floor * A_floor
W_corridor__typ = U_corridor * A_corridor

<IPython.core.display.Latex object>

This applies to levels 2, 3, and 4.

In [11]:
set_params_columns(3)

In [12]:
%%render params
W_floor__2 = W_floor__typ
W_floor__3 = W_floor__typ
W_floor__4 = W_floor__typ
W_corridor__2 = W_corridor__typ
W_corridor__3 = W_corridor__typ
W_corridor__4 = W_corridor__typ

<IPython.core.display.Latex object>

The roof weight is

In [13]:
%%render
W_roof = U_roof * A_level

<IPython.core.display.Latex object>

The exterior walls are divided in half to the floor above and below.

In [14]:
%%render
W_wall__2 = U_wall * (A_wall__2 / 2 + A_wall__1 / 2)
W_wall__3 = U_wall * (A_wall__3 / 2 + A_wall__2 / 2)
W_wall__4 = U_wall * (A_wall__4 / 2 + A_wall__3 / 2)
W_wall__roof = U_wall * (A_wall__4 / 2)

<IPython.core.display.Latex object>

## Level Weights

Each level's weight is

In [15]:
%%render 
W_level__2 = W_floor__2 + W_corridor__2 + W_wall__2
W_level__3 = W_floor__3 + W_corridor__3 + W_wall__3
W_level__4 = W_floor__4 + W_corridor__4 + W_wall__4
# be sure to add the weight of the truss later
W_level__roof = W_roof + W_wall__roof

<IPython.core.display.Latex object>

## Total Weight

The total weight of the structure is

In [16]:
%%render long
W_total = W_level__2 + W_level__3 + W_level__4 + W_level__roof

<IPython.core.display.Latex object>

# Seismic Equivalent Lateral Force (ELF) Analysis

We will be using the equivalent lateral force method prescribed by ASCE 7-16 Section 12.8. We will envelope the forces with a fully flexible diaphragm assumption and a fully rigid diaphragm assumption.

## Seismic Parameters

In [17]:
set_params_columns(6)

In [18]:
S_S = 0.887  # g
S_1 = 0.410  # g
seismic_params = SeismicParameters(s_1=S_1, s_s=S_S, risk_category=RiskCategory.II, site_class=SiteClass.D)

Seismic design parameters are obtained from the geotech report and ASCE 7-16 Sec. 11 and 12.

In [19]:
%%render params
R = 6.5
Omega_0 = 3.0
C_d = 4.0
rho = 1.0

<IPython.core.display.Latex object>

In [20]:
set_params_columns(5)

In [21]:
%%render params
I_e = seismic_params.i_e
S_S = seismic_params.s_s  # g
S_1 = seismic_params.s_1  # g
F_a = seismic_params.f_a
F_v = seismic_params.f_v
S_MS = seismic_params.s_ms  # g
S_M1 = seismic_params.s_m1  # g
S_DS = seismic_params.s_ds  # g
S_D1 = seismic_params.s_d1  # g
SDC = seismic_params.sdc
C_u = 1.4
T_L = 16  # s

<IPython.core.display.Latex object>

## Base Shear

We must calculate the period

In [22]:
set_params_columns(1)

In [23]:
%%render params
h_top__roof = (57 + 8 / 12) * ft
h_bottom__roof = 45.5 * ft
h_n = np.mean([h_top__roof, h_bottom__roof])  # ASCE 7-16 Sec. 11.2

<IPython.core.display.Latex object>

In [24]:
%%render
h_n__check = check_value(h_n, 65 * ft, "leq")

<IPython.core.display.Latex object>

We will approximate the building period.

In [25]:
h_n = float(h_n)  # to prevent weirdness with units

In [26]:
%%render
C_t = 0.02  # ASCE 7-16 Table 12.8-2
x = 0.75  # ASCE 7-16 Table 12.8-2
T_a = C_t * h_n**x  # ASCE 7-16 Eq. 12.8-7

<IPython.core.display.Latex object>

Because we are not using an automated analysis software, we can make the following assumption:

In [27]:
%%render
T = T_a  # ASCE 7-16 Sec. 12.8.2

<IPython.core.display.Latex object>

With the period, we can calculate the seismic response coefficient.

In [28]:
%%render
C_s = S_DS / (R / I_e)  # ASCE 7-16 Eq. 12.8-2

<IPython.core.display.Latex object>

We must check that $C_s$ does not exceed:

In [29]:
%%render long
# fmt: off
if T <= T_L:  C_s__check__max = S_D1 / (T * (R / I_e));              C_s__check__max = check_value(C_s, C_s__check__max, "leq")  # ASCE 7-16 Sec. 12.8.1.1
elif T > T_L: C_s__check__max = (S_D1 * T_L) / (T**2 * (R / I_e));   C_s__check__max = check_value(C_s, C_s__check__max, "leq")  # ASCE 7-16 Sec. 12.8.1.1
# fmt: on

<IPython.core.display.Latex object>

And we must also check that $C_s$ is not less than:

In [30]:
%%render
C_s__check__min__1 = 0.044 * S_DS * I_e  # ASCE 7-16 Eq. 12.8-5
C_s__check__min = check_value(C_s, C_s__check__min__1, "geq")

<IPython.core.display.Latex object>

as well as:

In [31]:
%%render
C_s__check__min = check_value(C_s, 0.01, "geq")  # ASCE 7-16 Eq. 12.8-5

<IPython.core.display.Latex object>

and finally:

In [32]:
%%render
# fmt: off
if S_1 >= 0.6:  C_s__check__min__3 = 0.5 * S_1 / (R / I_e); C_s__check__min = check_value(C_s, C_s__check__min__3, "geq")  # ASCE 7-16 Eq. 12.8-6
elif S_1 < 0.6: C_s__check__min = "\\text{skipping check}"  # ASCE 7-16 Eq. 12.8-6
# fmt: on

<IPython.core.display.Latex object>

And now, we can calculate the base shear, $V$

In [33]:
%%render
V = C_s * W_total  # ASCE 7-16 Eq. 12.8-1

<IPython.core.display.Latex object>

## Distribute the seismic forces

With the base shear calculated, we must now distribute the base shear on each level.

First, we must calculate the structural period exponent, $k$.

In [34]:
%%render
# fmt: off
if T <= 0.5: k = 1
elif T >= 2.5: k = 2
elif T > 0.5 and T < 2.5: k = 1 + ((T - 0.5) * (2 - 1)) / (2.5 - 0.5)
# fmt: on

<IPython.core.display.Latex object>

Now, we can distribute the forces per floor,

$$
F_x = C_{vx} V\;\;\text{(ASCE 7-16 Eq. 12.8-11)}
$$

As well as cumulatively from roof to ground.

$$
V_x = \sum_{i=x}^n F_i\;\; \text{(ASCE 7-16 Eq. 12.8-11)}
$$

In [35]:
# Load in basic data
seismic_loads = pd.DataFrame(
    {
        "Level": ["Ground", "Level 2", "Level 3", "Level 4", "Roof"],
        "floor height": [
            float(h_wall__1),
            float(h_wall__2),
            float(h_wall__3),
            float(h_wall__4),
            0,
        ],
        "floor weight": [
            0,
            float(W_level__2),
            float(W_level__3),
            float(W_level__4),
            float(W_level__roof),
        ],
    },
)
seismic_loads = seismic_loads.set_index("Level")

# Calc cumulative elevation
elevations = [0]
for height in seismic_loads["floor height"][:-1]:
    elevations.append(height + elevations[-1])
seismic_loads.insert(2, "elevation", elevations)

# Calc coefficients and story forces ASCE 7-16 Eq. 12.8-11 and 12.8-12
seismic_loads["wxhxk"] = seismic_loads["floor weight"] * seismic_loads["elevation"] ** float(k)
cumulative_wxhxk = seismic_loads["wxhxk"].sum()
seismic_loads["C_vx"] = seismic_loads["wxhxk"] / cumulative_wxhxk
seismic_loads["F_x"] = seismic_loads["C_vx"] * float(V)

# Calc story shear forces ASCE 7-16 Eq. 12.8-13
story_shears = [0]
for shear in reversed(list(seismic_loads["F_x"][1:])):
    story_shears.insert(0, shear + story_shears[0])
seismic_loads["V_x"] = story_shears

# Calc overturning moments
floor_moments = seismic_loads["V_x"] * seismic_loads["floor height"]
overturning_moments = [0]
for floor_moment in reversed(list(floor_moments[:-1])):
    overturning_moments.insert(0, overturning_moments[0] + floor_moment)
seismic_loads["OTM"] = overturning_moments

In [36]:
# Rename columns for display with LaTeX formatting for units and subscripts/superscripts
column_name_map = {
    "floor height": "$h_{floor}$ [ft]",
    "floor weight": "$w_x$ [kip]",
    "elevation": "$h_x$ [ft]",
    "wxhxk": "$w_x h_x^k$",
    "C_vx": "$C_{vx}$",
    "F_x": "$F_x$ [kip]",
    "V_x": "$V_x$ [kip]",
    "OTM": "OTM [kip-ft]",
}

Calculating for each floor leads to the following \autoref{tab:seismic_forces_calculation_table}

In [37]:
# Format the dataframe
# Reverse the order of the dataframe with '.loc[::-1]' so roof is the top row and ground is the bottom row, then apply formatting with the 'format_dataframe' function
display_table(
    dataframe=seismic_loads.iloc[::-1],
    column_names_filter_and_map=column_name_map,
    position_float="centering",
    caption="Seismic Forces Calculation Table",
    label="tab:seismic_forces_calculation_table",
)

,$h_{floor}$ [ft],$w_x$ [kip],$h_x$ [ft],$w_x h_x^k$,$C_{vx}$,$F_x$ [kip],$V_x$ [kip],OTM [kip-ft]
Level,,,,,,,,
Roof,0.0,421.490,42.5,17913.325,0.306127,72.387392,0.000000,0.000000
Level 4,10.0,613.642,32.5,19943.365,0.340819,80.590744,72.387392,723.873920
Level 3,10.5,615.454,22.0,13539.988,0.231389,54.714824,152.978136,2330.144343
Level 2,10.5,619.078,11.5,7119.397,0.121666,28.769342,207.692959,4510.920416
Ground,11.5,0.000,0.0,0.000,0.000000,0.000000,236.462302,7230.236885


# Diaphragm and Shear Wall Design

## Redundancy Factor

The redundancy factor, $\rho$, is either 1.3 or 1.0 according to ASCE 7-16 12.3.4.

No single story resists more than 35% of the shear force (see $C_{vx}$ in \autoref{tab:seismic_forces_calculation_table}), so all values of $\rho$ are $1.0$.

## Envelope Procedure

We will envelope the forces with a fully flexible diaphragm assumption and a fully rigid diaphragm assumption.

In [38]:
# Set up diaphragm dataframe
diaphragm_rigidity = ["Flexible", "Rigid"]
levels = ["Roof", "Level 4", "Level 3", "Level 2"]
walls = {
    "A-2": "N-S",
    "A-2.4": "N-S",
    "A-3.7": "N-S",
    "A-4": "N-S",
    "A-5": "N-S",
    "A-5.1": "N-S",
    "A-6": "N-S",
    "A-6.1": "N-S",
    "A-6.8": "N-S",
    "A-7": "N-S",
    "A-7.8": "N-S",
    "A-8": "N-S",
    "A-8.2": "N-S",
    "A-8.9": "N-S",
    "A-9": "N-S",
    # E-W
    "A-B_A-1": "E-W",
    "A-B_A-2.5": "E-W",
    "A-B_A-3.5": "E-W",
    "A-B_A-4": "E-W",
    "A-B_A-4.8": "E-W",
    "A-B_A-5.5": "E-W",
    "A-B_A-6.5": "E-W",
    "A-B_A-7.9": "E-W",
    "A-B_A-8.8": "E-W",
    "A-C_A-1.8": "E-W",
    "A-C_A-3.2": "E-W",
    "A-C_A-4.2": "E-W",
    "A-C_A-5.5": "E-W",
    "A-C_A-6.5": "E-W",
    "A-C_A-6.9": "E-W",
    "A-C_A-7.5": "E-W",
    "A-C_A-8.5": "E-W",
    "A-C_A-9.2": "E-W",
}

index = pd.MultiIndex.from_product(
    [diaphragm_rigidity, levels, walls.keys()],
    names=["Diaphragm Rigidity", "Level", "Wall"],
)

shear_walls = pd.DataFrame(index=index)
shear_walls["Direction"] = shear_walls.index.get_level_values("Wall").map(walls)
flexible = shear_walls.loc["Flexible"].copy()
flexible_ns = flexible[flexible["Direction"] == "N-S"].copy()
flexible_ew = flexible[flexible["Direction"] == "E-W"].copy()
rigid = shear_walls.loc["Rigid"].copy()
rigid_ns = rigid[rigid["Direction"] == "N-S"].copy()
rigid_ew = rigid[rigid["Direction"] == "E-W"].copy()

### Flexible Diaphragm Assumption

We will uniformly distribute the shear forces across the diaphragms based on tributary areas of each level. This removes some nuance in the distribution of floor weight distribution (corridors are 17% heavier than the rest of the floors, for instance), but the vast majority of the area of each level is typical floor weight (30 psf) instead of corridor weight (35 psf).

$$
f_{p,x} = F_x / A_{level}
$$

In [39]:
%%render
A_level
f_p__roof = seismic_loads.loc["Roof", "F_x"] * 1000 * lb / A_level
f_p__4 = seismic_loads.loc["Level 4", "F_x"] * 1000 * lb / A_level
f_p__3 = seismic_loads.loc["Level 3", "F_x"] * 1000 * lb / A_level
f_p__2 = seismic_loads.loc["Level 2", "F_x"] * 1000 * lb / A_level

<IPython.core.display.Latex object>

#### North-South Direction

Most of the shear walls are not in line which presents an issue: if we assume a diaphragm is the full depth, there will be an extremely long collector element without a shear wall on both sides. We will avoid this by just assuming the flexible diaphragm does not extend the full depth and instead is just tributary to the nearest wall (\autoref{fig:diaphragm_trib_area}). This means that there are essentially two diaphragms through the depth of the building. This is a rough approximation to ensure all the shear load is resolved in the shear walls. \

The walls that are close to inline (A-5 and A-5.1, A-6 and A-6.1, and A-9 and A-9.1) in reality will be treated as inline and have collectors to transfer the shear load from the diaphragm to those walls.

Proper detailing will be done to ensure continuity through the depth of the diaphragm.

![Shear walls (thick red lines) in plan view with their (colored) tributary areas.](images/shear_wall_tributary_area.png){width=60% #fig:diaphragm_trib_area}

Shear wall shear forces are tabulated in \autoref{tab:shear_wall_forces_roof_ns}, \autoref{tab:shear_wall_forces_level4_ns}, \autoref{tab:shear_wall_forces_level3_ns}, and \autoref{tab:shear_wall_forces_level2_ns}.

Shear wall sheathing will be assumed to have the following properties:

1. WSP Sheathing
2. 15/32" thickness
3. 10d common nails
4. Plywood panels
5. Framing uses Douglas-Fir-Larch (no specific gravity adjustment)

Notes: 

- Cumulative shear force, $F_{x,cum}$, is cumulative from the Roof downwards. For example, Level 3 includes the shear forces from the Roof, Level 4, and Level 3.

In [40]:
from structural_tools.seismic.timber.sheathing import (
    DesignMethod,
    LoadCase,
    Nail,
    PanelType,
    Sheathing,
    SheathingMaterial,
    get_sheathing_properties,
)

# Set baseline sheathing properties. We set the number of sheathed sides later
sheathing = Sheathing(
    sheathing_material=SheathingMaterial.WSP_SHEATHING,
    minimum_nominal_panel_thickness=15 / 32,
    nail=Nail.COMMON_10D,
    panel_type=PanelType.PLY,
)

In [41]:
# Plan dimensions
l_A1__A2 = (31 + 11 / 12) * ft
l_A2__A3 = (17 + 11 / 12) * ft
l_A3__A4 = (26 + 4 / 12) * ft
l_A4__A5 = (21 + 4 / 12) * ft
l_A5__A6 = (24 + 8 / 12) * ft
l_A6__A7 = (34 + 4 / 12) * ft
l_A7__A8 = (26 + 1 / 12) * ft
l_A8__A9 = (21 + 4 / 12) * ft
l_A9__A10 = (31 + 1 / 12) * ft

l_A1__A24 = (40 + 1 / 12) * ft
l_A24__A35 = (23 + 8 / 12) * ft
l_A35__A51 = (34 + 4 / 12) * ft
l_A51__A61 = (24 + 8 / 12) * ft
l_A61__A68 = (24 + 8 / 12) * ft
l_A68__A82 = (38 + 3 / 12) * ft
l_A82__A89 = (17 + 11 / 12) * ft
l_A89__A10 = (31 + 5 / 12) * ft

l_A37__A4 = (8 + 10 / 12) * ft
l_A3__A37 = l_A3__A4 - l_A37__A4
l_A78__A82 = (8 + 10 / 12) * ft
l_A68__A78 = l_A68__A82 - l_A78__A82

d_trib_ns = 67 / 2 * ft

In [42]:
# Wall tributary widths and lengths
t_w__A2 = l_A1__A2 + (l_A2__A3 + l_A3__A37) / 2
t_w__A24 = (l_A1__A24 - L_jog) + ((l_A24__A35 + l_A35__A51) / 2)
t_w__A37 = ((l_A2__A3 + l_A3__A37) / 2) + (l_A37__A4 / 2)
t_w__A4 = (l_A37__A4 / 2) + (l_A4__A5 / 2)
t_w__A5 = (l_A4__A5 / 2) + (l_A5__A6 / 2)
t_w__A51 = ((l_A24__A35 + l_A35__A51) / 2) + (l_A51__A61 / 2)
t_w__A6 = (l_A5__A6 / 2) + (l_A6__A7 / 2)
t_w__A61 = (l_A51__A61 / 2) + (l_A61__A68 / 2)
t_w__A68 = (l_A61__A68 / 2) + (l_A68__A78 / 2)
t_w__A7 = (l_A6__A7 / 2) + (l_A7__A8 / 2)
t_w__A78 = (l_A68__A78 / 2) + (l_A78__A82 / 2)
t_w__A8 = (l_A7__A8 / 2) + (l_A8__A9 / 2)
t_w__A82 = (l_A78__A82 / 2) + (l_A82__A89 / 2)
t_w__A89 = (l_A82__A89 / 2) + l_A89__A10
t_w__A9 = (l_A8__A9 / 2) + (l_A9__A10 - L_jog)

t_l__A2 = float(d_trib_ns)
t_l__A24 = float(d_trib_ns)
t_l__A37 = float(d_trib_ns)
t_l__A4 = float(d_trib_ns)
t_l__A5 = float(d_trib_ns)
t_l__A51 = float(d_trib_ns)
t_l__A6 = float(d_trib_ns)
t_l__A61 = float(d_trib_ns)
t_l__A68 = float(d_trib_ns)
t_l__A7 = float(d_trib_ns)
t_l__A78 = float(d_trib_ns)
t_l__A8 = float(d_trib_ns)
t_l__A82 = float(d_trib_ns)
t_l__A89 = float(d_trib_ns)
t_l__A9 = float(d_trib_ns)

In [43]:
flexible_ns.loc["Roof", "tributary weight"] = float(f_p__roof)
flexible_ns.loc["Level 4", "tributary weight"] = float(f_p__4)
flexible_ns.loc["Level 3", "tributary weight"] = float(f_p__3)
flexible_ns.loc["Level 2", "tributary weight"] = float(f_p__2)

# Shear wall height is for the floor below
flexible_ns.loc["Roof", "wall height"] = float(seismic_loads.loc["Level 4", "floor height"])
flexible_ns.loc["Level 4", "wall height"] = float(seismic_loads.loc["Level 3", "floor height"])
flexible_ns.loc["Level 3", "wall height"] = float(seismic_loads.loc["Level 2", "floor height"])
flexible_ns.loc["Level 2", "wall height"] = float(seismic_loads.loc["Ground", "floor height"])

# Add wall tributary widths and wall lengths to df
l_aa__ab = 30 + 9 / 12
l_ac__ad = 30 + 9 / 12
l_stairwall = 20
wall_tributary = {
    "A-2": {"trib_width": float(t_w__A2), "trib_depth": t_l__A2, "wall_length": l_aa__ab},
    "A-2.4": {"trib_width": float(t_w__A24), "trib_depth": t_l__A24, "wall_length": l_ac__ad},
    "A-3.7": {"trib_width": float(t_w__A37), "trib_depth": t_l__A37, "wall_length": l_stairwall},
    "A-4": {"trib_width": float(t_w__A4), "trib_depth": t_l__A4, "wall_length": l_stairwall},
    "A-5": {"trib_width": float(t_w__A5), "trib_depth": t_l__A5, "wall_length": l_aa__ab},
    "A-5.1": {"trib_width": float(t_w__A51), "trib_depth": t_l__A51, "wall_length": l_ac__ad},
    "A-6": {"trib_width": float(t_w__A6), "trib_depth": t_l__A6, "wall_length": l_aa__ab},
    "A-6.1": {"trib_width": float(t_w__A61), "trib_depth": t_l__A61, "wall_length": l_ac__ad},
    "A-6.8": {"trib_width": float(t_w__A68), "trib_depth": t_l__A68, "wall_length": l_ac__ad},
    "A-7": {"trib_width": float(t_w__A7), "trib_depth": t_l__A7, "wall_length": l_aa__ab},
    "A-7.8": {"trib_width": float(t_w__A78), "trib_depth": t_l__A78, "wall_length": l_stairwall},
    "A-8": {"trib_width": float(t_w__A8), "trib_depth": t_l__A8, "wall_length": l_aa__ab},
    "A-8.2": {"trib_width": float(t_w__A82), "trib_depth": t_l__A82, "wall_length": l_stairwall},
    "A-8.9": {"trib_width": float(t_w__A89), "trib_depth": t_l__A89, "wall_length": l_ac__ad},
    "A-9": {"trib_width": float(t_w__A9), "trib_depth": t_l__A9, "wall_length": l_aa__ab},
}

flexible_ns["wall length"] = flexible_ns.index.get_level_values("Wall").map(
    {key: value["wall_length"] for key, value in wall_tributary.items()}
)
flexible_ns["tributary width"] = flexible_ns.index.get_level_values("Wall").map(
    {key: value["trib_width"] for key, value in wall_tributary.items()}
)
flexible_ns["tributary depth"] = flexible_ns.index.get_level_values("Wall").map(
    {key: value["trib_depth"] for key, value in wall_tributary.items()}
)
flexible_ns["tributary depth"] = float(d_trib_ns)
flexible_ns["tributary area"] = flexible_ns["tributary depth"] * flexible_ns["tributary width"]

# Ensure that the calculated areas from earlier and tributary areas match
trib_area = flexible_ns["tributary area"].sum() / 4
assert np.isclose(trib_area, float(A_level)), (
    f"Shear wall tributary areas ({trib_area}) and level area ({float(A_level)}) do not match up. Ensure you are including all areas in your calculations"
)

# Automatically extract the shear line by splitting at "_" and taking the first section
flexible_ns["line"] = flexible_ns.index.get_level_values("Wall").str.split("_").str[0].astype(str)

# Get tributary area and wall length for the shear line
flexible_ns["line tributary area"] = flexible_ns.groupby(["Level", "line"])["tributary area"].transform("sum")
flexible_ns["line wall length"] = flexible_ns.groupby(["Level", "line"])["wall length"].transform("sum")

flexible_ns["line shear demand"] = flexible_ns["line tributary area"] * flexible_ns["tributary weight"]
flexible_ns["line cumulative shear demand"] = flexible_ns.groupby(level="Wall")["line shear demand"].cumsum()
flexible_ns["line unit shear"] = flexible_ns["line cumulative shear demand"] / flexible_ns["line wall length"]

flexible_ns["shear demand"] = flexible_ns["line unit shear"] * flexible_ns["wall length"]
flexible_ns["cumulative shear demand"] = flexible_ns.groupby(level="Wall")["shear demand"].cumsum()
flexible_ns["unit shear demand"] = flexible_ns["cumulative shear demand"] / flexible_ns["wall length"]
asd_load_factor = 0.7
flexible_ns["adjusted unit shear demand"] = flexible_ns["unit shear demand"] * asd_load_factor

flexible_ns["sheathed sides"] = 1
flexible_ns["aspect ratio"] = flexible_ns["wall height"] / flexible_ns["wall length"]
flexible_ns

Direction  tributary weight  wall height  wall length  \
Level   Wall                                                          
Roof    A-2         N-S          4.697430         10.0        30.75   
        A-2.4       N-S          4.697430         10.0        30.75   
        A-3.7       N-S          4.697430         10.0        20.00   
        A-4         N-S          4.697430         10.0        20.00   
        A-5         N-S          4.697430         10.0        30.75   
        A-5.1       N-S          4.697430         10.0        30.75   
        A-6         N-S          4.697430         10.0        30.75   
        A-6.1       N-S          4.697430         10.0        30.75   
        A-6.8       N-S          4.697430         10.0        30.75   
        A-7         N-S          4.697430         10.0        30.75   
        A-7.8       N-S          4.697430         10.0        20.00   
        A-8         N-S          4.697430         10.0        30.75   
        A-8.2       N-S          4.697430         10.0        20.00   
        A-8.9       N-S          4.697430         10.0        30.75   
        A-9         N-S          4.697430         10.0        30.75   
Level 4 A-2         N-S          5.229769         10.5        30.75   
        A-2.4       N-S          5.229769         10.5        30.75   
        A-3.7       N-S          5.229769         10.5        20.00   
        A-4         N-S          5.229769         10.5        20.00   
        A-5         N-S          5.229769         10.5        30.75   
        A-5.1       N-S          5.229769         10.5        30.75   
        A-6         N-S          5.229769         10.5        30.75   
        A-6.1       N-S          5.229769         10.5        30.75   
        A-6.8       N-S          5.229769         10.5        30.75   
        A-7         N-S          5.229769         10.5        30.75   
        A-7.8       N-S          5.229769         10.5        20.00   
        A-8         N-S          5.229769         10.5        30.75   
        A-8.2       N-S          5.229769         10.5        20.00   
        A-8.9       N-S          5.229769         10.5        30.75   
        A-9         N-S          5.229769         10.5        30.75   
Level 3 A-2         N-S          3.550605         10.5        30.75   
        A-2.4       N-S          3.550605         10.5        30.75   
        A-3.7       N-S          3.550605         10.5        20.00   
        A-4         N-S          3.550605         10.5        20.00   
        A-5         N-S          3.550605         10.5        30.75   
        A-5.1       N-S          3.550605         10.5        30.75   
        A-6         N-S          3.550605         10.5        30.75   
        A-6.1       N-S          3.550605         10.5        30.75   
        A-6.8       N-S          3.550605         10.5        30.75   
        A-7         N-S          3.550605         10.5        30.75   
        A-7.8       N-S          3.550605         10.5        20.00   
        A-8         N-S          3.550605         10.5        30.75   
        A-8.2       N-S          3.550605         10.5        20.00   
        A-8.9       N-S          3.550605         10.5        30.75   
        A-9         N-S          3.550605         10.5        30.75   
Level 2 A-2         N-S          1.866927         11.5        30.75   
        A-2.4       N-S          1.866927         11.5        30.75   
        A-3.7       N-S          1.866927         11.5        20.00   
        A-4         N-S          1.866927         11.5        20.00   
        A-5         N-S          1.866927         11.5        30.75   
        A-5.1       N-S          1.866927         11.5        30.75   
        A-6         N-S          1.866927         11.5        30.75   
        A-6.1       N-S          1.866927         11.5        30.75   
        A-6.8       N-S          1.866927         11.5        30.75   
        A-7         N-S          1.866

#### Check aspect ratios

We must check the aspect ratios of the shear walls to ensure we don't need to apply the aspect ratio factor in SDPWS Sec. 4.3.3.2. The maximum aspect ratio of all N-S walls is:

In [44]:
%%render
ratio_max = check_value(flexible_ns["aspect ratio"].max(), 2, "<")

<IPython.core.display.Latex object>

In [45]:
if "OK" in ratio_max:
    display_text(r"$\therefore$ All aspect ratios are OK")
else:
    display_text("The aspect ratio knockdown factor must be applied.")

'$\\therefore$ All aspect ratios are OK'

In [46]:
flexible_ns[["adjusted unit shear capacity", "nail spacing", "shear stiffness", "shear dcr"]] = (
    get_sheathing_properties(flexible_ns[["adjusted unit shear demand", "sheathed sides"]], sheathing=sheathing)
)

shear_dcr_check_value = 1

# Assign 2-sided sheathing to walls requiring it, then recalc capacity for just those walls (more efficient than passing the full wall dataframe)
if (flexible_ns["shear dcr"] > shear_dcr_check_value).any():
    flexible_ns.loc[flexible_ns["shear dcr"] > shear_dcr_check_value, "sheathed sides"] = 2
    flexible_ns.loc[
        flexible_ns["shear dcr"] > shear_dcr_check_value,
        ["adjusted unit shear capacity", "nail spacing", "shear stiffness", "shear dcr"],
    ] = get_sheathing_properties(
        flexible_ns.loc[
            flexible_ns["shear dcr"] > shear_dcr_check_value, ["adjusted unit shear demand", "sheathed sides"]
        ],
        sheathing=sheathing,
    )

In [47]:
# Update dataframe
# flexible = flexible.reindex(columns=flexible_ns.columns)
# # flexible
# flexible.update(flexible_ns)

In [48]:
# Rename columns for display with LaTeX formatting for units and subscripts/superscripts
column_name_map = {
    "tributary weight": "$w_{trib}$ [psf]",
    "tributary area": "$A$ [ft$^2$]",
    "shear demand": "$F_x$ [lbf]",
    "cumulative shear demand": "$F_{x,cum}$ [lbf]",
    "wall length": "$l_{wall}$ [ft]",
    "unit shear demand": "$v$ [plf]",
    "adjusted unit shear demand": "$0.7 v$ [lbf]",
    "adjusted unit shear capacity": "$v_{cap}$ [plf]",
    "sheathed sides": "sides",
    "nail spacing": "$s_{nail}$ [in]",
}

In [49]:
def highlight_insufficient_capacity(row):
    styles = [""] * len(row)

    if row["$0.7 v$ [plf]"] > row["$v_{cap}$ [plf]"]:
        demand_idx = row.index.get_loc("$0.7 v$ [plf]")
        capacity_idx = row.index.get_loc("$v_{cap}$ [plf]")

        styles[demand_idx] = "background-color: red; font-weight: bold"
        styles[capacity_idx] = "background-color: red; font-weight: bold"

    return styles

In [50]:
display_table(
    dataframe=flexible_ns,
    levels="Roof",
    column_names_filter_and_map=column_name_map,
    position_float="centering",
    caption="Shear wall forces NS - Roof",
    label="tab:shear_wall_forces_roof_ns",
)
display_table(
    dataframe=flexible_ns,
    levels="Level 4",
    column_names_filter_and_map=column_name_map,
    position_float="centering",
    caption="Shear wall forces NS - Level 4",
    label="tab:shear_wall_forces_level4_ns",
)
display_table(
    dataframe=flexible_ns,
    levels="Level 3",
    column_names_filter_and_map=column_name_map,
    position_float="centering",
    caption="Shear wall forces NS - Level 3",
    label="tab:shear_wall_forces_level3_ns",
)
display_table(
    dataframe=flexible_ns,
    levels="Level 2",
    column_names_filter_and_map=column_name_map,
    position_float="centering",
    caption="Shear wall forces NS - Level 2",
    label="tab:shear_wall_forces_level2_ns",
)

,$w_{trib}$ [psf],$A$ [ft$^2$],$F_x$ [lbf],"$F_{x,cum}$ [lbf]",$l_{wall}$ [ft],$v$ [plf],$0.7 v$ [lbf],$v_{cap}$ [plf],sides,$s_{nail}$ [in]
Wall,,,,,,,,,,
A-2,4.69743,1662.437500,7809.183325,7809.183325,30.75,253.957181,177.770027,310.714286,1,6.0
A-2.4,4.69743,2146.791667,10084.402984,10084.402984,30.75,327.948065,229.563645,310.714286,1,6.0
A-3.7,4.69743,741.187500,3481.676193,3481.676193,20.00,174.083810,121.858667,310.714286,1,6.0
A-4,4.69743,505.291667,2373.572094,2373.572094,20.00,118.678605,83.075023,310.714286,1,6.0
A-5,4.69743,770.500000,3619.369601,3619.369601,30.75,117.703076,82.392154,310.714286,1,6.0
A-5.1,4.69743,1384.666667,6504.374356,6504.374356,30.75,211.524369,148.067059,310.714286,1,6.0
A-6,4.69743,988.250000,4642.234923,4642.234923,30.75,150.966989,105.676893,310.714286,1,6.0
A-6.1,4.69743,826.333333,3881.642761,3881.642761,30.75,126.232285,88.362599,310.714286,1,6.0
A-6.8,4.69743,905.895833,4255.382013,4255.382013,30.75,138.386407,96.870485,310.714286,1,6.0


,$w_{trib}$ [psf],$A$ [ft$^2$],$F_x$ [lbf],"$F_{x,cum}$ [lbf]",$l_{wall}$ [ft],$v$ [plf],$0.7 v$ [lbf],$v_{cap}$ [plf],sides,$s_{nail}$ [in]
Wall,,,,,,,,,,
A-2,5.229769,1662.437500,16503.347775,24312.531099,30.75,790.651418,553.455993,600.000000,1,3.0
A-2.4,5.229769,2146.791667,21311.627941,31396.030924,30.75,1021.009136,714.706395,769.642857,1,2.0
A-3.7,5.229769,741.187500,7357.915758,10839.591951,20.00,541.979598,379.385718,460.714286,1,4.0
A-4,5.229769,505.291667,5016.130894,7389.702987,20.00,369.485149,258.639605,310.714286,1,6.0
A-5,5.229769,770.500000,7648.906777,11268.276379,30.75,366.448012,256.513609,310.714286,1,6.0
A-5.1,5.229769,1384.666667,13745.861455,20250.235811,30.75,658.544254,460.980978,600.000000,1,3.0
A-6,5.229769,988.250000,9810.554345,14452.789268,30.75,470.009407,329.006585,460.714286,1,4.0
A-6.1,5.229769,826.333333,8203.175384,12084.818145,30.75,393.002216,275.101551,310.714286,1,6.0
A-6.8,5.229769,905.895833,8993.008149,13248.390162,30.75,430.841957,301.589370,310.714286,1,6.0


,$w_{trib}$ [psf],$A$ [ft$^2$],$F_x$ [lbf],"$F_{x,cum}$ [lbf]",$l_{wall}$ [ft],$v$ [plf],$0.7 v$ [lbf],$v_{cap}$ [plf],sides,$s_{nail}$ [in]
Wall,,,,,,,,,,
A-2,3.550605,1662.437500,22406.006745,46718.537845,30.75,1519.302044,1063.511431,1200.000000,2,3.0
A-2.4,3.550605,2146.791667,28934.037258,60330.068182,30.75,1961.953437,1373.367406,1539.285714,2,2.0
A-3.7,3.550605,741.187500,9989.579834,20829.171785,20.00,1041.458589,729.021012,769.642857,1,2.0
A-4,3.550605,505.291667,6810.222033,14199.925021,20.00,709.996251,496.997376,600.000000,1,3.0
A-5,3.550605,770.500000,10384.647963,21652.924341,30.75,704.160141,492.912099,600.000000,1,3.0
A-5.1,3.550605,1384.666667,18662.265904,38912.501715,30.75,1265.447210,885.813047,921.428571,2,4.0
A-6,3.550605,988.250000,13319.439778,27772.229046,30.75,903.161920,632.213344,769.642857,1,2.0
A-6.1,3.550605,826.333333,11137.158685,23221.976830,30.75,755.186238,528.630367,600.000000,1,3.0
A-6.8,3.550605,905.895833,12209.486463,25457.876626,30.75,827.898427,579.528899,600.000000,1,3.0


,$w_{trib}$ [psf],$A$ [ft$^2$],$F_x$ [lbf],"$F_{x,cum}$ [lbf]",$l_{wall}$ [ft],$v$ [plf],$0.7 v$ [lbf],$v_{cap}$ [plf],sides,$s_{nail}$ [in]
Wall,,,,,,,,,,
A-2,1.866927,1662.437500,25509.655919,72228.193764,30.75,2348.884350,1644.219045,1539.285714,2,2.0
A-2.4,1.866927,2146.791667,32941.940221,93272.008404,30.75,3033.236046,2123.265232,1539.285714,2,2.0
A-3.7,1.866927,741.187500,11373.322664,32202.494449,20.00,1610.124722,1127.087306,1200.000000,2,3.0
A-4,1.866927,505.291667,7753.564603,21953.489624,20.00,1097.674481,768.372137,769.642857,1,2.0
A-5,1.866927,770.500000,11823.115086,33476.039427,30.75,1088.651689,762.056182,769.642857,1,2.0
A-5.1,1.866927,1384.666667,21247.337256,60159.838970,30.75,1956.417527,1369.492269,1539.285714,2,2.0
A-6,1.866927,988.250000,15164.430219,42936.659265,30.75,1396.314122,977.419886,1200.000000,2,3.0
A-6.1,1.866927,826.333333,12679.862556,35901.839386,30.75,1167.539492,817.277645,921.428571,2,4.0
A-6.8,1.866927,905.895833,13900.727701,39358.604326,30.75,1279.954612,895.968229,921.428571,2,4.0


#### East-West Direction

Most likely, the shear walls in the E-W direction will be governed by a rigid diaphragm assumption. 

Shear wall shear forces are tabulated in \autoref{tab:shear_wall_forces_roof_ew}, \autoref{tab:shear_wall_forces_level4_ew}, \autoref{tab:shear_wall_forces_level3_ew}, and \autoref{tab:shear_wall_forces_level2_ew}. \

In [51]:
# Plan dimensions
l_AA__AB = (30 + 9 / 12) * ft
l_AB__AC = (5 + 6 / 12) * ft
l_AC__AD = (30 + 9 / 12) * ft

d_trib_ew = L_floor

In [52]:
# Wall tributary widths and lengths
t_w__AB = l_AA__AB + l_AB__AC / 2
t_w__AC = l_AB__AC / 2 + l_AC__AD

t_l__AB__1 = 33.75
t_l__AB__2 = 22.25
t_l__AB__3 = 16
t_l__AB__4 = 19
t_l__AB__5 = 16
t_l__AB__6 = 34.5
t_l__AB__7 = 28.25
t_l__AB__8 = 28.5
t_l__AB__9 = 31.75

t_l__AC__1 = 44
t_l__AC__2 = 31
t_l__AC__3 = 27.5
t_l__AC__4 = 31
t_l__AC__5 = 15.5
t_l__AC__6 = 12.34
t_l__AC__7 = 19
t_l__AC__8 = 21.33
t_l__AC__9 = 28.33

In [53]:
# Wall segment lengths
l_AB__1 = 21
l_AB__2 = 13.25
l_AB__3 = 9
l_AB__4 = 12
l_AB__5 = 11
l_AB__6 = 29.75
l_AB__7 = 12.75
l_AB__8 = 16.25
l_AB__9 = 13.75

l_AC__1 = 17.25
l_AC__2 = 23
l_AC__3 = 22.5
l_AC__4 = 26
l_AC__5 = 11.25
l_AC__6 = 7
l_AC__7 = 6.75
l_AC__8 = 10.25
l_AC__9 = 25.25

In [54]:
flexible_ew.loc["Roof", "tributary weight"] = float(f_p__roof)
flexible_ew.loc["Level 4", "tributary weight"] = float(f_p__4)
flexible_ew.loc["Level 3", "tributary weight"] = float(f_p__3)
flexible_ew.loc["Level 2", "tributary weight"] = float(f_p__2)

# Shear wall height is for the floor below
flexible_ew.loc["Roof", "wall height"] = float(seismic_loads.loc["Level 4", "floor height"])
flexible_ew.loc["Level 4", "wall height"] = float(seismic_loads.loc["Level 3", "floor height"])
flexible_ew.loc["Level 3", "wall height"] = float(seismic_loads.loc["Level 2", "floor height"])
flexible_ew.loc["Level 2", "wall height"] = float(seismic_loads.loc["Ground", "floor height"])

# Add wall tributary widths and wall lengths to df
wall_tributary = {
    "A-B_A-1": {"trib_width": float(t_w__AB), "trib_depth": t_l__AB__1, "wall_length": l_AB__1},
    "A-B_A-2.5": {"trib_width": float(t_w__AB), "trib_depth": t_l__AB__2, "wall_length": l_AB__2},
    "A-B_A-3.5": {"trib_width": float(t_w__AB), "trib_depth": t_l__AB__3, "wall_length": l_AB__3},
    "A-B_A-4": {"trib_width": float(t_w__AB), "trib_depth": t_l__AB__4, "wall_length": l_AB__4},
    "A-B_A-4.8": {"trib_width": float(t_w__AB), "trib_depth": t_l__AB__5, "wall_length": l_AB__5},
    "A-B_A-5.5": {"trib_width": float(t_w__AB), "trib_depth": t_l__AB__6, "wall_length": l_AB__6},
    "A-B_A-6.5": {"trib_width": float(t_w__AB), "trib_depth": t_l__AB__7, "wall_length": l_AB__7},
    "A-B_A-7.9": {"trib_width": float(t_w__AB), "trib_depth": t_l__AB__8, "wall_length": l_AB__8},
    "A-B_A-8.8": {"trib_width": float(t_w__AB), "trib_depth": t_l__AB__9, "wall_length": l_AB__9},
    "A-C_A-1.8": {"trib_width": float(t_w__AC), "trib_depth": t_l__AC__1, "wall_length": l_AC__1},
    "A-C_A-3.2": {"trib_width": float(t_w__AC), "trib_depth": t_l__AC__2, "wall_length": l_AC__2},
    "A-C_A-4.2": {"trib_width": float(t_w__AC), "trib_depth": t_l__AC__3, "wall_length": l_AC__3},
    "A-C_A-5.5": {"trib_width": float(t_w__AC), "trib_depth": t_l__AC__4, "wall_length": l_AC__4},
    "A-C_A-6.5": {"trib_width": float(t_w__AC), "trib_depth": t_l__AC__5, "wall_length": l_AC__5},
    "A-C_A-6.9": {"trib_width": float(t_w__AC), "trib_depth": t_l__AC__6, "wall_length": l_AC__6},
    "A-C_A-7.5": {"trib_width": float(t_w__AC), "trib_depth": t_l__AC__7, "wall_length": l_AC__7},
    "A-C_A-8.5": {"trib_width": float(t_w__AC), "trib_depth": t_l__AC__8, "wall_length": l_AC__8},
    "A-C_A-9.2": {"trib_width": float(t_w__AC), "trib_depth": t_l__AC__9, "wall_length": l_AC__9},
}

flexible_ew["wall length"] = flexible_ew.index.get_level_values("Wall").map(
    {key: value["wall_length"] for key, value in wall_tributary.items()}
)
flexible_ew["tributary width"] = flexible_ew.index.get_level_values("Wall").map(
    {key: value["trib_width"] for key, value in wall_tributary.items()}
)
flexible_ew["tributary depth"] = flexible_ew.index.get_level_values("Wall").map(
    {key: value["trib_depth"] for key, value in wall_tributary.items()}
)
flexible_ew["tributary area"] = flexible_ew["tributary depth"] * flexible_ew["tributary width"]

# Ensure that the calculated areas from earlier and tributary areas match
trib_area = flexible_ew["tributary area"].sum() / 4
assert np.isclose(trib_area, float(A_level)), (
    f"Shear wall tributary areas ({trib_area}) and level area ({float(A_level)}) do not match up.\n\
    Ensure you are including all areas in your calculations"
)

# Automatically extract the shear line by splitting at "_" and taking the first section
flexible_ew["line"] = flexible_ew.index.get_level_values("Wall").str.split("_").str[0].astype(str)

# Get tributary area and wall length for the shear line
flexible_ew["line tributary area"] = flexible_ew.groupby(["Level", "line"])["tributary area"].transform("sum")
flexible_ew["line wall length"] = flexible_ew.groupby(["Level", "line"])["wall length"].transform("sum")

flexible_ew["line shear demand"] = flexible_ew["line tributary area"] * flexible_ew["tributary weight"]
flexible_ew["line unit shear"] = flexible_ew["line shear demand"] / flexible_ew["line wall length"]
flexible_ew["line cumulative shear demand"] = flexible_ew.groupby(level="Wall")["line shear demand"].cumsum()
flexible_ew["line cumulative unit shear"] = (
    flexible_ew["line cumulative shear demand"] / flexible_ew["line wall length"]
)

flexible_ew["shear demand"] = flexible_ew["line unit shear"] * flexible_ew["wall length"]
flexible_ew["cumulative shear demand"] = flexible_ew.groupby(level="Wall")["shear demand"].cumsum()
flexible_ew["unit shear demand"] = flexible_ew["cumulative shear demand"] / flexible_ew["wall length"]
asd_load_factor = 0.7
flexible_ew["adjusted unit shear demand"] = flexible_ew["unit shear demand"] * asd_load_factor

flexible_ew["sheathed sides"] = 1
flexible_ew["aspect ratio"] = flexible_ew["wall height"] / flexible_ew["wall length"]
flexible_ew

Direction  tributary weight  wall height  wall length  \
Level   Wall                                                              
Roof    A-B_A-1         E-W          4.697430         10.0        21.00   
        A-B_A-2.5       E-W          4.697430         10.0        13.25   
        A-B_A-3.5       E-W          4.697430         10.0         9.00   
        A-B_A-4         E-W          4.697430         10.0        12.00   
        A-B_A-4.8       E-W          4.697430         10.0        11.00   
        A-B_A-5.5       E-W          4.697430         10.0        29.75   
        A-B_A-6.5       E-W          4.697430         10.0        12.75   
        A-B_A-7.9       E-W          4.697430         10.0        16.25   
        A-B_A-8.8       E-W          4.697430         10.0        13.75   
        A-C_A-1.8       E-W          4.697430         10.0        17.25   
        A-C_A-3.2       E-W          4.697430         10.0        23.00   
        A-C_A-4.2       E-W          4.697430         10.0        22.50   
        A-C_A-5.5       E-W          4.697430         10.0        26.00   
        A-C_A-6.5       E-W          4.697430         10.0        11.25   
        A-C_A-6.9       E-W          4.697430         10.0         7.00   
        A-C_A-7.5       E-W          4.697430         10.0         6.75   
        A-C_A-8.5       E-W          4.697430         10.0        10.25   
        A-C_A-9.2       E-W          4.697430         10.0        25.25   
Level 4 A-B_A-1         E-W          5.229769         10.5        21.00   
        A-B_A-2.5       E-W          5.229769         10.5        13.25   
        A-B_A-3.5       E-W          5.229769         10.5         9.00   
        A-B_A-4         E-W          5.229769         10.5        12.00   
        A-B_A-4.8       E-W          5.229769         10.5        11.00   
        A-B_A-5.5       E-W          5.229769         10.5        29.75   
        A-B_A-6.5       E-W          5.229769         10.5        12.75   
        A-B_A-7.9       E-W          5.229769         10.5        16.25   
        A-B_A-8.8       E-W          5.229769         10.5        13.75   
        A-C_A-1.8       E-W          5.229769         10.5        17.25   
        A-C_A-3.2       E-W          5.229769         10.5        23.00   
        A-C_A-4.2       E-W          5.229769         10.5        22.50   
        A-C_A-5.5       E-W          5.229769         10.5        26.00   
        A-C_A-6.5       E-W          5.229769         10.5        11.25   
        A-C_A-6.9       E-W          5.229769         10.5         7.00   
        A-C_A-7.5       E-W          5.229769         10.5         6.75   
        A-C_A-8.5       E-W          5.229769         10.5        10.25   
        A-C_A-9.2       E-W          5.229769         10.5        25.25   
Level 3 A-B_A-1         E-W          3.550605         10.5        21.00   
        A-B_A-2.5       E-W          3.550605         10.5        13.25   
        A-B_A-3.5       E-W          3.550605         10.5         9.00   
        A-B_A-4         E-W          3.550605         10.5        12.00   
        A-B_A-4.8       E-W          3.550605         10.5        11.00   
        A-B_A-5.5       E-W          3.550605         10.5        29.75   
        A-B_A-6.5       E-W          3.550605         10.5        12.75   
        A-B_A-7.9       E-W          3.550605         10.5        16.25   
        A-B_A-8.8       E-W          3.550605         10.5        13.75   
        A-C_A-1.8       E-W          3.550605         10.5        17.25   
        A-C_A-3.2       E-W          3.550605         10.5        23.00   
        A-C_A-4.2       E-W          3.550605         10.5        22.50   
        A-C_A-5.5       E-W          3.550605         10.5        26.00   
        A-C_A-6.5       E-W          3.550605         10.5        11.25   
        A-C_A-6.9       E-W          3.550605         10.5         7.00   
        A-C_A-7.5       E-W          3.5506

#### Check aspect ratios

We will also check the aspect ratios of the E-W walls.

In [55]:
%%render
ratio_max = check_value(flexible_ew["aspect ratio"].max(), 2, "<")

<IPython.core.display.Latex object>

$\therefore$ All aspect ratios are OK

In [56]:
# using sheathing from the NS calc in memory

flexible_ew[["adjusted unit shear capacity", "nail spacing", "shear stiffness", "shear dcr"]] = (
    get_sheathing_properties(
        flexible_ew[["adjusted unit shear demand", "sheathed sides"]],
        sheathing=sheathing,
    )
)

# Assign 2-sided sheathing to walls requiring it, then recalc capacity for just those walls (more efficient than passing the full wall dataframe)
if (flexible_ew["shear dcr"] > shear_dcr_check_value).any():
    flexible_ew.loc[flexible_ew["shear dcr"] > shear_dcr_check_value, "sheathed sides"] = 2
    flexible_ew.loc[
        flexible_ew["shear dcr"] > shear_dcr_check_value,
        ["adjusted unit shear capacity", "nail spacing", "shear stiffness", "shear dcr"],
    ] = get_sheathing_properties(
        flexible_ew.loc[
            flexible_ew["shear dcr"] > shear_dcr_check_value,
            ["adjusted unit shear demand", "sheathed sides"],
        ],
        sheathing=sheathing,
    )

In [57]:
# Update dataframe
flexible.update(flexible_ew)

In [58]:
# Rename columns for display with LaTeX formatting for units and subscripts/superscripts
column_name_map = {
    "tributary weight": "$W_{trib}$ [psf]",
    "tributary area": "$A$ [ft$^2$]",
    "shear demand": "$F_x$ [lbf]",
    "cumulative shear demand": "$F_{x,cum}$ [lbf]",
    "wall length": "$l_{wall}$ [ft]",
    "unit shear demand": "$v$ [plf]",
    "adjusted unit shear demand": "$0.7 v$ [plf]",
    "adjusted unit shear capacity": "$v_{cap}$ [plf]",
    "sheathed sides": "sides",
    "nail spacing": "$s_{nail}$ [in]",
}

In [59]:
display_table(
    dataframe=flexible_ew,
    levels="Roof",
    column_names_filter_and_map=column_name_map,
    position_float="centering",
    caption="Shear wall forces EW - Roof",
    label="tab:shear_wall_forces_roof_ew",
)
display_table(
    dataframe=flexible_ew,
    levels="Level 4",
    column_names_filter_and_map=column_name_map,
    position_float="centering",
    caption="Shear wall forces EW - Level 4",
    label="tab:shear_wall_forces_level4_ew",
)
display_table(
    dataframe=flexible_ew,
    levels="Level 3",
    column_names_filter_and_map=column_name_map,
    position_float="centering",
    caption="Shear wall forces EW - Level 3",
    label="tab:shear_wall_forces_level3_ew",
)
display_table(
    dataframe=flexible_ew,
    levels="Level 2",
    column_names_filter_and_map=column_name_map,
    position_float="centering",
    caption="Shear wall forces EW - Level 2",
    label="tab:shear_wall_forces_level2_ew",
)

,$W_{trib}$ [psf],$A$ [ft$^2$],$F_x$ [lbf],"$F_{x,cum}$ [lbf]",$l_{wall}$ [ft],$v$ [plf],$0.7 v$ [plf],$v_{cap}$ [plf],sides,$s_{nail}$ [in]
Wall,,,,,,,,,,
A-B_A-1,4.69743,1130.625,5477.964802,5477.964802,21.00,260.855467,182.598827,310.714286,1,6.0
A-B_A-2.5,4.69743,745.375,3456.334935,3456.334935,13.25,260.855467,182.598827,310.714286,1,6.0
A-B_A-3.5,4.69743,536.000,2347.699201,2347.699201,9.00,260.855467,182.598827,310.714286,1,6.0
A-B_A-4,4.69743,636.500,3130.265601,3130.265601,12.00,260.855467,182.598827,310.714286,1,6.0
A-B_A-4.8,4.69743,536.000,2869.410134,2869.410134,11.00,260.855467,182.598827,310.714286,1,6.0
A-B_A-5.5,4.69743,1155.750,7760.450136,7760.450136,29.75,260.855467,182.598827,310.714286,1,6.0
A-B_A-6.5,4.69743,946.375,3325.907201,3325.907201,12.75,260.855467,182.598827,310.714286,1,6.0
A-B_A-7.9,4.69743,954.750,4238.901335,4238.901335,16.25,260.855467,182.598827,310.714286,1,6.0
A-B_A-8.8,4.69743,1063.625,3586.762668,3586.762668,13.75,260.855467,182.598827,310.714286,1,6.0


,$W_{trib}$ [psf],$A$ [ft$^2$],$F_x$ [lbf],"$F_{x,cum}$ [lbf]",$l_{wall}$ [ft],$v$ [plf],$0.7 v$ [plf],$v_{cap}$ [plf],sides,$s_{nail}$ [in]
Wall,,,,,,,,,,
A-B_A-1,5.229769,1130.625,6098.758969,11576.723771,21.00,551.272561,385.890792,460.714286,1,4.0
A-B_A-2.5,5.229769,745.375,3848.026492,7304.361427,13.25,551.272561,385.890792,460.714286,1,4.0
A-B_A-3.5,5.229769,536.000,2613.753844,4961.453045,9.00,551.272561,385.890792,460.714286,1,4.0
A-B_A-4,5.229769,636.500,3485.005125,6615.270726,12.00,551.272561,385.890792,460.714286,1,4.0
A-B_A-4.8,5.229769,536.000,3194.588031,6063.998166,11.00,551.272561,385.890792,460.714286,1,4.0
A-B_A-5.5,5.229769,1155.750,8639.908539,16400.358675,29.75,551.272561,385.890792,460.714286,1,4.0
A-B_A-6.5,5.229769,946.375,3702.817945,7028.725147,12.75,551.272561,385.890792,460.714286,1,4.0
A-B_A-7.9,5.229769,954.750,4719.277773,8958.179108,16.25,551.272561,385.890792,460.714286,1,4.0
A-B_A-8.8,5.229769,1063.625,3993.235039,7579.997707,13.75,551.272561,385.890792,460.714286,1,4.0


,$W_{trib}$ [psf],$A$ [ft$^2$],$F_x$ [lbf],"$F_{x,cum}$ [lbf]",$l_{wall}$ [ft],$v$ [plf],$0.7 v$ [plf],$v_{cap}$ [plf],sides,$s_{nail}$ [in]
Wall,,,,,,,,,,
A-B_A-1,3.550605,1130.625,4140.581254,15717.305024,21.00,748.443096,523.910167,600.0,1,3.0
A-B_A-2.5,3.550605,745.375,2612.509600,9916.871027,13.25,748.443096,523.910167,600.0,1,3.0
A-B_A-3.5,3.550605,536.000,1774.534823,6735.987868,9.00,748.443096,523.910167,600.0,1,3.0
A-B_A-4,3.550605,636.500,2366.046431,8981.317157,12.00,748.443096,523.910167,600.0,1,3.0
A-B_A-4.8,3.550605,536.000,2168.875895,8232.874060,11.00,748.443096,523.910167,600.0,1,3.0
A-B_A-5.5,3.550605,1155.750,5865.823443,22266.182118,29.75,748.443096,523.910167,600.0,1,3.0
A-B_A-6.5,3.550605,946.375,2513.924333,9542.649479,12.75,748.443096,523.910167,600.0,1,3.0
A-B_A-7.9,3.550605,954.750,3204.021208,12162.200317,16.25,748.443096,523.910167,600.0,1,3.0
A-B_A-8.8,3.550605,1063.625,2711.094868,10291.092576,13.75,748.443096,523.910167,600.0,1,3.0


,$W_{trib}$ [psf],$A$ [ft$^2$],$F_x$ [lbf],"$F_{x,cum}$ [lbf]",$l_{wall}$ [ft],$v$ [plf],$0.7 v$ [plf],$v_{cap}$ [plf],sides,$s_{nail}$ [in]
Wall,,,,,,,,,,
A-B_A-1,1.866927,1130.625,2177.139430,17894.444454,21.00,852.116403,596.481482,600.0,1,3.0
A-B_A-2.5,1.866927,745.375,1373.671307,11290.542334,13.25,852.116403,596.481482,600.0,1,3.0
A-B_A-3.5,1.866927,536.000,933.059756,7669.047623,9.00,852.116403,596.481482,600.0,1,3.0
A-B_A-4,1.866927,636.500,1244.079674,10225.396831,12.00,852.116403,596.481482,600.0,1,3.0
A-B_A-4.8,1.866927,536.000,1140.406368,9373.280428,11.00,852.116403,596.481482,600.0,1,3.0
A-B_A-5.5,1.866927,1155.750,3084.280859,25350.462977,29.75,852.116403,596.481482,600.0,1,3.0
A-B_A-6.5,1.866927,946.375,1321.834654,10864.484133,12.75,852.116403,596.481482,600.0,1,3.0
A-B_A-7.9,1.866927,954.750,1684.691226,13846.891542,16.25,852.116403,596.481482,600.0,1,3.0
A-B_A-8.8,1.866927,1063.625,1425.507960,11716.600536,13.75,852.116403,596.481482,600.0,1,3.0


### Rigid Diaphragm Assumption

With a rigid diaphragm assumption, we now must calculate the following in each direction:

1. The relative stiffnesses of the shear walls to obtain a center of rigidity.
2. The center of mass of the diaphragm with the center of rigidity to obtain an eccentricity.

The relative stiffnesses of the shear walls can be calculated by using the deflection equation from NDS SDPWS Sec. 4.3.4

\begin{align*}
k &= \frac{F}{\delta},\, \text{assume }F = \text{induced load demand in the shear wall} \\
\delta_{sw} &= \frac{8vh^3}{EAb} + \frac{vh}{1000 G_a} + \frac{h \Delta_A}{b}\quad \text{(SDPWS Eq. 4.3-1)}
\end{align*}

where

- $b =$ the shear wall length [ft]
- $\Delta_A =$ vertical deformation of the wall overturning anchorage system + vertical compression deformation [in]
- $E =$ modulus of elasticity of end posts [psi]
- $A =$ area of end post cross-section [in$^2$]
- $G_a =$ apparent shear wall shear stiffness from nail slip and panel shear deformation [kip/in]
- $h =$ shear wall height [ft]
- $v =$ unit shear force induced by the design load [plf]
- $\delta_{sw} =$ maximum shear wall deflection determined by elastic analysis [in]

Note that for the following calculations, we will conservatively use the unit shear force induced by the nominal load, not the design load, for the value of $v$. This is because the stiffnesses obtained from this use will allow us to use the stiffnesses for deflection and story drift calculations

We will assume that the end posts are (2) 2x4 Douglas Fir No. 2; this means:

In [67]:
%%render
E_endpost = 1_600_000 * psi
A_endpost = 2 * 5.25 * inch**2

<IPython.core.display.Latex object>

We will assume a continuous tiedown rod system. A conservative first-pass value for $\Delta_A$ can be:

In [68]:
%%render
Delta_A = 0.25 * inch

<IPython.core.display.Latex object>

#### North-South Direction

In [72]:
rigid_ns["unit shear demand"] = flexible_ns["unit shear demand"]
rigid_ns["wall length"] = flexible_ns["wall length"]
rigid_ns["wall height"] = flexible_ns["wall height"]
rigid_ns["shear stiffness"] = flexible_ns["shear stiffness"]
rigid_ns["nail spacing"] = flexible_ns["nail spacing"]
rigid_ns["end post youngs modulus"] = float(E_endpost)
rigid_ns["end post area"] = float(A_endpost)
rigid_ns["Delta_A"] = float(Delta_A)

rigid_ns["delta_sw"] = (
    8
    * rigid_ns["unit shear demand"]
    * rigid_ns["wall height"] ** 3
    / (rigid_ns["end post youngs modulus"] * rigid_ns["end post area"] * rigid_ns["wall length"])
    + rigid_ns["unit shear demand"] * rigid_ns["wall height"] / (1000 * rigid_ns["shear stiffness"])
    + rigid_ns["wall height"] * rigid_ns["Delta_A"] / rigid_ns["wall length"]
)

##### Shear Wall Rigidities

With the deflections of each shear wall, along with the force demand of each shear wall, we can obtain the stiffnesses (and therefore, relative stiffnesses) of all the walls.

$$
k = \frac{F_x}{\delta_{sw}}
$$

In [73]:
rigid_ns["cumulative shear demand"] = flexible_ns["cumulative shear demand"]
rigid_ns["wall stiffness"] = rigid_ns["cumulative shear demand"] / rigid_ns["delta_sw"]
rigid_ns["total level stiffness"] = rigid_ns.groupby("Level")["wall stiffness"].transform("sum")
rigid_ns["total level cumulative shear"] = rigid_ns.groupby("Level")["cumulative shear demand"].transform("sum")
rigid_ns["relative wall stiffness"] = rigid_ns["wall stiffness"] / rigid_ns["total level stiffness"]
rigid_ns["direct shear force"] = rigid_ns["total level cumulative shear"] * rigid_ns["relative wall stiffness"]
rigid_ns

Direction  unit shear demand  wall length  wall height  \
Level   Wall                                                           
Roof    A-2         N-S         253.957181        30.75         10.0   
        A-2.4       N-S         327.948065        30.75         10.0   
        A-3.7       N-S         174.083810        20.00         10.0   
        A-4         N-S         118.678605        20.00         10.0   
        A-5         N-S         117.703076        30.75         10.0   
        A-5.1       N-S         211.524369        30.75         10.0   
        A-6         N-S         150.966989        30.75         10.0   
        A-6.1       N-S         126.232285        30.75         10.0   
        A-6.8       N-S         138.386407        30.75         10.0   
        A-7         N-S         154.591903        30.75         10.0   
        A-7.8       N-S         150.479225        20.00         10.0   
        A-8         N-S         121.327990        30.75         10.0   
        A-8.2       N-S         105.237105        20.00         10.0   
        A-8.9       N-S         206.620074        30.75         10.0   
        A-9         N-S         188.069046        30.75         10.0   
Level 4 A-2         N-S         790.651418        30.75         10.5   
        A-2.4       N-S        1021.009136        30.75         10.5   
        A-3.7       N-S         541.979598        20.00         10.5   
        A-4         N-S         369.485149        20.00         10.5   
        A-5         N-S         366.448012        30.75         10.5   
        A-5.1       N-S         658.544254        30.75         10.5   
        A-6         N-S         470.009407        30.75         10.5   
        A-6.1       N-S         393.002216        30.75         10.5   
        A-6.8       N-S         430.841957        30.75         10.5   
        A-7         N-S         481.294944        30.75         10.5   
        A-7.8       N-S         468.490839        20.00         10.5   
        A-8         N-S         377.733549        30.75         10.5   
        A-8.2       N-S         327.637384        20.00         10.5   
        A-8.9       N-S         643.275587        30.75         10.5   
        A-9         N-S         585.520194        30.75         10.5   
Level 3 A-2         N-S        1519.302044        30.75         10.5   
        A-2.4       N-S        1961.953437        30.75         10.5   
        A-3.7       N-S        1041.458589        20.00         10.5   
        A-4         N-S         709.996251        20.00         10.5   
        A-5         N-S         704.160141        30.75         10.5   
        A-5.1       N-S        1265.447210        30.75         10.5   
        A-6         N-S         903.161920        30.75         10.5   
        A-6.1       N-S         755.186238        30.75         10.5   
        A-6.8       N-S         827.898427        30.75         10.5   
        A-7         N-S         924.848012        30.75         10.5   
        A-7.8       N-S         900.243865        20.00         10.5   
        A-8         N-S         725.846232        30.75         10.5   
        A-8.2       N-S         629.582311        20.00         10.5   
        A-8.9       N-S        1236.107204        30.75         10.5   
        A-9         N-S        1125.125443        30.75         10.5   
Level 2 A-2         N-S        2348.884350        30.75         11.5   
        A-2.4       N-S        3033.236046        30.75         11.5   
        A-3.7       N-S        1610.124722        20.00         11.5   
        A-4         N-S        1097.674481        20.00         11.5   
        A-5         N-S        1088.651689        30.75         11.5   
        A-5.1       N-S        1956.417527        30.75         11.5   
        A-6         N-S        1396.314122        30.75         11.5   
        A-6.1       N-S        1167.539492        30.75         11.5   
        A-6.8       N-S        1279.954612        30.7

#### East-West Direction

In [74]:
rigid_ew["unit shear demand"] = flexible_ew["unit shear demand"]
rigid_ew["wall length"] = flexible_ew["wall length"]
rigid_ew["wall height"] = flexible_ew["wall height"]
rigid_ew["shear stiffness"] = flexible_ew["shear stiffness"]
rigid_ew["nail spacing"] = flexible_ew["nail spacing"]
rigid_ew["end post youngs modulus"] = float(E_endpost)
rigid_ew["end post area"] = float(A_endpost)
rigid_ew["Delta_A"] = float(Delta_A)

rigid_ew["delta_sw"] = (
    8
    * rigid_ew["unit shear demand"]
    * rigid_ew["wall height"] ** 3
    / (rigid_ew["end post youngs modulus"] * rigid_ew["end post area"] * rigid_ew["wall length"])
    + rigid_ew["unit shear demand"] * rigid_ew["wall height"] / (1000 * rigid_ew["shear stiffness"])
    + rigid_ew["wall height"] * rigid_ew["Delta_A"] / rigid_ew["wall length"]
)

##### Shear Wall Rigidities

With the deflections of each shear wall, along with the force demand of each shear wall, we can obtain the stiffnesses (and therefore, relative stiffnesses) of all the walls.

$$
k = \frac{F_x}{\delta_{sw}}
$$

In [75]:
rigid_ew["cumulative shear demand"] = flexible_ew["cumulative shear demand"]
rigid_ew["wall stiffness"] = rigid_ew["cumulative shear demand"] / rigid_ew["delta_sw"]
rigid_ew["total level stiffness"] = rigid_ew.groupby("Level")["wall stiffness"].transform("sum")
rigid_ew["total level cumulative shear"] = rigid_ew.groupby("Level")["cumulative shear demand"].transform("sum")
rigid_ew["relative wall stiffness"] = rigid_ew["wall stiffness"] / rigid_ew["total level stiffness"]
rigid_ew["direct shear force"] = rigid_ew["total level cumulative shear"] * rigid_ew["relative wall stiffness"]
rigid_ew

Direction  unit shear demand  wall length  wall height  \
Level   Wall                                                               
Roof    A-B_A-1         E-W         260.855467        21.00         10.0   
        A-B_A-2.5       E-W         260.855467        13.25         10.0   
        A-B_A-3.5       E-W         260.855467         9.00         10.0   
        A-B_A-4         E-W         260.855467        12.00         10.0   
        A-B_A-4.8       E-W         260.855467        11.00         10.0   
...                     ...                ...          ...          ...   
Level 2 A-C_A-6.5       E-W        2242.950715        11.25         11.5   
        A-C_A-6.9       E-W        2242.950715         7.00         11.5   
        A-C_A-7.5       E-W        2242.950715         6.75         11.5   
        A-C_A-8.5       E-W        2242.950715        10.25         11.5   
        A-C_A-9.2       E-W        2242.950715        25.25         11.5   

                   shear stiffness  nail spacing  end post youngs modulus  \
Level   Wall                                                                
Roof    A-B_A-1               14.0           6.0                1600000.0   
        A-B_A-2.5             14.0           6.0                1600000.0   
        A-B_A-3.5             14.0           6.0                1600000.0   
        A-B_A-4               14.0           6.0                1600000.0   
        A-B_A-4.8             14.0           6.0                1600000.0   
...                            ...           ...                      ...   
Level 2 A-C_A-6.5             23.0           2.0                1600000.0   
        A-C_A-6.9             23.0           2.0                1600000.0   
        A-C_A-7.5             23.0           2.0                1600000.0   
        A-C_A-8.5             23.0           2.0                1600000.0   
        A-C_A-9.2             23.0           2.0                1600000.0   

                   end post area  Delta_A  delta_sw  cumulative shear demand  \
Level   Wall                                                                   
Roof    A-B_A-1             10.5     0.25  0.311288              5477.964802   
        A-B_A-2.5           10.5     0.25  0.384379              3456.334935   
        A-B_A-3.5           10.5     0.25  0.477905              2347.699201   
        A-B_A-4             10.5     0.25  0.405010              3130.265601   
        A-B_A-4.8           10.5     0.25  0.424891              2869.410134   
...                          ...      ...       ...                      ...   
Level 2 A-C_A-6.5           10.5     0.25  1.521422             25233.195548   
        A-C_A-6.9           10.5     0.25  1.764247             15700.655008   
        A-C_A-7.5           10.5     0.25  1.788054             15139.917329   
        A-C_A-8.5           10.5     0.25  1.560442             22990.244833   
        A-C_A-9.2           10.5     0.25  1.299670             56634.505563   

                   wall stiffness  total level stiffness  \
Level   Wall                                               
Roof    A-B_A-1      17597.736038          219295.417863   
        A-B_A-2.5     8991.987060          219295.417863   
        A-B_A-3.5     4912.481060          219295.417863   
        A-B_A-4       7728.858616          219295.417863   
        A-B_A-4.8     6753.293142          219295.417863   
...                           ...                    ...   
Level 2 A-C_A-6.5    16585.266783          465989.290793   
        A-C_A-6.9     8899.350412          465989.290793   
        A-C_A-7.5     8467.261094          465989.290793   
        A-C_A-8.5    14733.166117          465989.290793   
        A-C_A-9.2    43576.080591          465989.290793   

                   total level cumulative shear  relative wall stiffness  \
Level   Wall                                                               
Roof    A-B_A-1                    72387.392027      

##### Story Drift

We can check the story drift at each shear wall line. This is conservative but all-encompassing.

The story drift is defined as:

$$
\delta_x = \frac{C_d \delta_{xe}}{I_e}\quad \text{(ASCE 7-16 Eq. 12.8-15)}
$$

We can get the elastic story displacement, $\delta_{xe}$, by getting a displacement for each shear line (which will be conservative). Since we got a stiffness using $\delta_{sw}$, we can add those stiffnesses up for each line then get a displacement with $u = F/k$.

The story drift can't exceed the allowable story drift, $\Delta_a$, which is defined in ASCE 7-16 Table 12.12-1. In our case:

- Structure is not masonry shear wall
- Four stories or less above the base
- Interior walls, partitions, ceilings, and exterior wall systems are designed to accommodate story drifts.

Therefore:

$$
\Delta_a = 0.025 h_{sx}
$$

where $h_{sx}$ is the story height below level $x$.

In [76]:
rigid_ew["story drift"] = rigid_ew["delta_sw"] * C_d / I_e
rigid_ew["Delta_a"] = 0.025 * rigid_ew["wall height"] * 12
rigid_ew["story drift check"] = rigid_ew["story drift"] <= rigid_ew["Delta_a"]
rigid_ew["story drift dcr"] = rigid_ew["story drift"] / rigid_ew["Delta_a"]

The ratio of the story drift, $\Delta_x$, and the the maximum story drift, $\Delta_a$, must not exceed 1

$$
\text{ratio}_{drift} = \frac{\Delta_x}{\Delta_a} \leq 1
$$

In [77]:
%%render
ratio_drift__max = rigid_ew["story drift dcr"].max()
check_Delta = check_value(ratio_drift__max, 1, inequality="<=")

<IPython.core.display.Latex object>

In [78]:
if rigid_ew["story drift check"].all():
    display_text("All story drifts are OK")

In [79]:
display_table(
    rigid_ew.loc[
        rigid_ew["story drift dcr"] > 1,
        [
            "shear stiffness",
            "story drift dcr",
        ],
    ],
    caption="\\textbf{\\color{red}SHEAR WALLS WITH INSUFFICIENT STIFFNESS}",
)

shear stiffness  story drift dcr
Level   Wall                                       
Level 4 A-B_A-3.5             19.0         1.003450
        A-C_A-6.9             19.0         1.081510
        A-C_A-7.5             19.0         1.101943
Level 3 A-B_A-1               19.0         1.305887
        A-B_A-2.5             19.0         1.429155
        A-B_A-3.5             19.0         1.586886
        A-B_A-4               19.0         1.463949
        A-B_A-4.8             19.0         1.497477
        A-B_A-5.5             19.0         1.243902
        A-B_A-6.5             19.0         1.442254
        A-B_A-7.9             19.0         1.367490
        A-B_A-8.8             19.0         1.417009
        A-C_A-1.8             19.0         1.270202
        A-C_A-3.2             19.0         1.207175
        A-C_A-4.2             19.0         1.211376
        A-C_A-5.5             19.0         1.185357
        A-C_A-6.5             19.0         1.404660
        A-C_A-6.9             19.0         1.639361
        A-C_A-7.5             19.0         1.662371
        A-C_A-8.5             19.0         1.442374
        A-C_A-9.2             19.0         1.190326
Level 2 A-B_A-1               23.0         1.653861
        A-B_A-2.5             23.0         1.803129
        A-B_A-3.5             23.0         1.994129
        A-B_A-4               23.0         1.845262
        A-B_A-4.8             23.0         1.885862
        A-B_A-5.5             23.0         1.578801
        A-B_A-6.5             23.0         1.818991
        A-B_A-7.9             23.0         1.728458
        A-B_A-8.8             23.0         1.788421
        A-C_A-1.8             23.0         1.602679
        A-C_A-3.2             23.0         1.527074
        A-C_A-4.2             23.0         1.532115
        A-C_A-5.5             23.0         1.500904
        A-C_A-6.5             23.0         1.763968
        A-C_A-6.9             23.0         2.045504
        A-C_A-7.5             23.0         2.073106
        A-C_A-8.5             23.0         1.809208
        A-C_A-9.2             23.0         1.506863